# Lab 01: Probability, Bayesian Thinking & the Python Toolkit

**Course:** 12062 Algorithmic Robotics  
**Lab Session:** Week 2 (First Lab of the Course)  
**Duration:** 2 hours  
**Prerequisites:** Lectures 1 (Probability Foundations) and 2 (Bayesian Inference)

---

## Goals

1. **Reinforce Weeks 1–2 lectures** — hands-on computational experience with probability and Bayesian inference
2. **Introduce the Python toolkit** — NumPy arrays, matplotlib plotting, and scipy.stats
3. **Build toward SLAM** — the patterns you practise here (log-odds, 2D grids, Gaussian PDFs, Bayesian updates) directly prepare you for the occupancy grid and scan matching code in Weeks 5–6

### Difficulty Tags

- `[WARM-UP]` — Everyone should complete these
- `[CORE]` — Expected completion for all students
- `[CHALLENGE]` — Stretch for fast students
- `[STRETCH]` — Optional, only for students who finish early

---

## Section 0: Setup & Imports `[WARM-UP]` — 5 min

Run the cell below to import the libraries we'll use throughout this lab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12

# Verify setup
print(f"NumPy version:  {np.__version__}")
print(f"SciPy version:  {stats.__version__ if hasattr(stats, '__version__') else 'OK'}")
print("All imports successful — you're ready to go!")

---

## Section 1: NumPy & Matplotlib Warm-Up `[WARM-UP]` — 15 min

If you're already comfortable with NumPy, breeze through this quickly. If arrays and plotting are new to you, take your time here — every later exercise builds on these skills.

### Exercise 1.1: Array Creation and Basic Operations (5 min)

**Context:** You are analysing the daily high temperatures (in Celsius) recorded at a Canberra weather station for one week.

**(a)** Create a NumPy array of the 7 temperature values: `[22, 25, 19, 28, 31, 27, 24]`

In [ ]:
# (a) Create the temperature array
temps = # YOUR CODE HERE
print(temps)

**(b)** Compute the mean and standard deviation using `np.mean()` and `np.std()`. Print both with descriptive labels.

In [ ]:
# (b) Compute mean and standard deviation
mean_temp = # YOUR CODE HERE
std_temp = # YOUR CODE HERE
print(f"Mean temperature: {mean_temp:.1f} °C")
print(f"Standard deviation: {std_temp:.1f} °C")

**(c)** Create a boolean mask of "hot days" (above 26°C) and use it to extract only those values.

> **Hint:** Boolean indexing lets you filter an array: `arr[arr > value]` returns only elements that satisfy the condition.

In [ ]:
# (c) Boolean masking for hot days
hot_mask = # YOUR CODE HERE — create a boolean condition
hot_days = # YOUR CODE HERE — use the mask to filter temps
print(f"Hot days: {hot_days}")
print(f"Number of hot days: {len(hot_days)}")

**(d)** Use `np.linspace(15, 35, 100)` to create 100 evenly-spaced values from 15 to 35. Print the first 5 values.

In [ ]:
# (d) Create evenly-spaced values
x = # YOUR CODE HERE
print(f"First 5 values: {x[:5]}")

### Exercise 1.2: Your First Plots (10 min)

**Context:** Visualise the temperature data from Exercise 1.1.

**(a)** Create a bar chart of daily temperatures. Label x-axis with day names, y-axis with "Temperature (°C)". Add a title.

> **Hint:** Use `plt.bar(categories, values)` where categories is a list of strings.

In [ ]:
# (a) Bar chart of temperatures
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# YOUR CODE HERE — create the bar chart with labels and title

plt.show()

**(b)** Recreate the bar chart but add a horizontal dashed red line at the mean temperature.

> **Hint:** `plt.axhline(y=value, color='red', linestyle='--', label='...')` adds a horizontal line.

In [ ]:
# (b) Bar chart with mean line

# YOUR CODE HERE — bar chart + horizontal mean line + legend

plt.show()

**(c)** Create a 1×2 subplot figure: left panel shows the bar chart, right panel shows a line plot of the same data.

> **Hint:** Use `fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))` to create side-by-side panels. Then use `ax1.bar(...)` and `ax2.plot(...)` instead of `plt.bar(...)` and `plt.plot(...)`.

In [ ]:
# (c) Side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# YOUR CODE HERE — bar chart on ax1, line plot on ax2

plt.tight_layout()
plt.show()

**(d)** Create a line plot with `plt.fill_between()` to shade the region within one standard deviation of the mean.

> **Hint:** `ax.fill_between(x_positions, lower_bound, upper_bound, alpha=0.2)` shades a horizontal band. The lower bound is `mean - std` and the upper bound is `mean + std`.

In [ ]:
# (d) Line plot with ±1 std dev shading
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = range(len(days))

# YOUR CODE HERE — line plot, mean line, fill_between for ±1 std dev

ax.set_xticks(x_pos)
ax.set_xticklabels(days)
ax.set_ylabel('Temperature (°C)')
ax.set_title('Temperature with ±1 Standard Deviation')
ax.legend()
plt.show()

---

## Section 2: Discrete Distributions with NumPy `[CORE]` — 20 min

### Exercise 2.1: PMFs as NumPy Arrays (8 min)

**Context:** A factory produces light bulbs. Quality testing shows the number of defects per bulb follows this distribution:

| Defects (x) | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| P(X = x) | 0.60 | 0.25 | 0.10 | 0.04 | 0.01 |

**(a)** Store the values and probabilities as two NumPy arrays.

In [ ]:
# (a) Define the PMF
x = # YOUR CODE HERE — array of defect counts
pmf = # YOUR CODE HERE — array of probabilities

**(b)** Verify this is a valid PMF: check all values are non-negative (Axiom 1) and they sum to 1 (Axiom 2).

> **Hint:** `np.all(condition)` returns True if every element satisfies the condition. `np.isclose(a, b)` checks if two floats are approximately equal (safer than `==` for floating point).

In [ ]:
# (b) Verify PMF validity
print(f"All non-negative? {np.all(pmf >= 0)}")              # Axiom 1
print(f"Sums to 1?       {np.isclose(np.sum(pmf), 1.0)}")   # Axiom 2

**(c)** Compute E[X], E[X²], Var(X), and the standard deviation.

> **Recall:** E[X] = Σ x·p(x),  Var(X) = E[X²] − (E[X])²
>
> **Hint:** In NumPy, `x * pmf` multiplies element-by-element (no loops needed!), and `np.sum(...)` adds them up.

In [ ]:
# (c) Compute expectation and variance
E_X = # YOUR CODE HERE — np.sum(x * pmf)
E_X2 = # YOUR CODE HERE — np.sum(x**2 * pmf)
Var_X = # YOUR CODE HERE — E[X²] - (E[X])²
std_X = # YOUR CODE HERE — square root of variance

print(f"E[X]   = {E_X:.4f}")
print(f"E[X²]  = {E_X2:.4f}")
print(f"Var(X) = {Var_X:.4f}  (using E[X²] - E[X]²)")
print(f"Std(X) = {std_X:.4f}")

**(d)** Create a bar chart of the PMF. Add a vertical red dashed line at E[X] and annotate with the mean and standard deviation.

In [ ]:
# (d) Visualise the PMF

# YOUR CODE HERE — bar chart with vertical mean line and text annotation

plt.xlabel('Number of Defects')
plt.ylabel('P(X = x)')
plt.title('Defect Count PMF')
plt.show()

### Exercise 2.2: Joint Distribution as a 2D Array (12 min)

**Context:** A sports analytics company tracks two variables for basketball players:
- **Position:** Guard (0), Forward (1), Centre (2)
- **Height category:** Short (0), Medium (1), Tall (2)

The joint PMF from a dataset of 500 players:

|  | Short | Medium | Tall |
|---|---|---|---|
| **Guard** | 0.15 | 0.20 | 0.05 |
| **Forward** | 0.03 | 0.18 | 0.14 |
| **Centre** | 0.01 | 0.06 | 0.18 |

**(a)** Create the joint distribution as a 2D NumPy array. Verify it sums to 1.0.

In [ ]:
# (a) Create the 2D joint distribution array
joint = # YOUR CODE HERE — np.array with 3 rows and 3 columns

print(f"Sum of joint distribution: {np.sum(joint):.2f}")

**(b)** Compute the marginal distributions by summing across the appropriate axis.

> **Hint:** `np.sum(array, axis=1)` sums across columns (gives one value per row). `np.sum(array, axis=0)` sums down rows (gives one value per column). Think about which axis corresponds to which variable.

In [ ]:
# (b) Compute marginal distributions

# Marginal of Position: sum across columns (axis=1)
p_position = # YOUR CODE HERE
print(f"P(Position): Guard={p_position[0]:.2f}, Forward={p_position[1]:.2f}, Centre={p_position[2]:.2f}")

# Marginal of Height: sum across rows (axis=0)
p_height = # YOUR CODE HERE
print(f"P(Height):   Short={p_height[0]:.2f}, Medium={p_height[1]:.2f}, Tall={p_height[2]:.2f}")

**(c)** Compute the conditional distribution P(Height | Position = Guard). Verify it sums to 1.

> **Recall:** P(Height | Guard) = P(Height, Guard) / P(Guard). Extract the Guard row with `joint[0, :]` and divide by the Guard marginal.

In [ ]:
# (c) Conditional distribution P(Height | Guard)
p_height_given_guard = # YOUR CODE HERE — extract Guard row and normalise

print(f"P(Height | Guard): Short={p_height_given_guard[0]:.3f}, "
      f"Medium={p_height_given_guard[1]:.3f}, Tall={p_height_given_guard[2]:.3f}")
print(f"Sum: {np.sum(p_height_given_guard):.3f}")

**(d)** Visualise the joint distribution as a **heatmap** using `plt.imshow()`. Add a colour bar and annotate each cell with its probability value.

> **Hint:** `plt.imshow(array, cmap='Blues')` displays a 2D array as a colour grid. Use `plt.colorbar()` to add a scale. To annotate cells, use a nested loop with `ax.text(col, row, text, ha='center', va='center')`.

In [ ]:
# (d) Heatmap of joint distribution
positions = ['Guard', 'Forward', 'Centre']
heights = ['Short', 'Medium', 'Tall']

fig, ax = plt.subplots(figsize=(6, 5))

# YOUR CODE HERE — imshow, colorbar, axis labels, and cell annotations

plt.tight_layout()
plt.show()

> **Looking Ahead:** In Week 5, you will build an **occupancy grid** — a much larger version of this heatmap (1000×700 cells) that represents the robot's belief about which areas are occupied by walls. The same `plt.imshow()` + `plt.colorbar()` pattern displays it.

---

## Section 3: Continuous Distributions & Gaussians `[CORE]` — 20 min

### Exercise 3.1: Plotting the Gaussian PDF (10 min)

**Context:** The height of adult women in Australia follows approximately N(μ = 164, σ = 7) cm.

**(a)** Create an x-axis of 200 points from 140 to 190 cm. Compute and plot the Gaussian PDF.

> **Important:** In `scipy.stats.norm`, the `scale` parameter is σ (standard deviation), **not** σ² (variance).  
> Usage: `stats.norm.pdf(x, loc=mean, scale=std_dev)`

In [ ]:
# (a) Plot the Gaussian PDF
x = np.linspace(140, 190, 200)
pdf = # YOUR CODE HERE — use stats.norm.pdf()

plt.plot(x, pdf, 'b-', linewidth=2)
plt.xlabel('Height (cm)')
plt.ylabel('Probability Density')
plt.title('Height Distribution: N(164, 7²)')
plt.show()

**(b)** Shade the region within one standard deviation of the mean (157 to 171 cm). Annotate with "68%".

> **Hint:** Create a boolean mask: `mask = (x >= 157) & (x <= 171)`, then use `plt.fill_between(x[mask], pdf[mask], alpha=0.3)`.

In [ ]:
# (b) Shade the ±1σ region
plt.plot(x, pdf, 'b-', linewidth=2)

# YOUR CODE HERE — create mask, fill_between, add "68%" text annotation

plt.xlabel('Height (cm)')
plt.ylabel('Probability Density')
plt.title('The 68-95-99.7 Rule')
plt.legend()
plt.show()

**(c)** Create a 1×3 subplot comparing three Gaussians with different standard deviations:
- σ = 3 (very confident) — green
- σ = 7 (moderate) — blue  
- σ = 12 (very uncertain) — orange

Left panel: overlay all three PDFs. Middle panel: shade the 95% region (±2σ) for each. Right panel: plot the CDFs.

> **Hint:** Use a loop over `sigmas = [3, 7, 12]` to avoid repetitive code. CDF is `stats.norm.cdf(x, loc=164, scale=sigma)`.

In [ ]:
# (c) Compare three Gaussians
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

sigmas = [3, 7, 12]
colours = ['green', 'blue', 'orange']
labels = ['σ=3 (very confident)', 'σ=7 (moderate)', 'σ=12 (very uncertain)']

# YOUR CODE HERE — Left: overlay PDFs, Middle: shade 95% regions, Right: CDFs

plt.tight_layout()
plt.show()

**(d)** Compute the exact probability P(160 ≤ X ≤ 170) using the CDF.

> **Recall:** P(a ≤ X ≤ b) = CDF(b) − CDF(a)

In [ ]:
# (d) Exact probability via CDF
p_range = # YOUR CODE HERE — CDF(170) - CDF(160)

print(f"P(160 ≤ X ≤ 170) = {p_range:.4f}  (exact via CDF)")
print(f"68-95-99.7 approximation: ~68% for ±1σ = ±7 cm")

### Exercise 3.2: Sampling and the Central Limit Theorem (10 min)

> **Note:** This exercise is skippable if you are behind schedule. Skip ahead to Section 4 if needed.

**Context:** The Central Limit Theorem (CLT) says: when you sum many independent random variables, the result approaches a Gaussian — regardless of the original distribution. This is why sensor noise in robotics is modelled as Gaussian.

Let's verify the CLT experimentally using dice rolls.

**(a)** Simulate 10,000 single die rolls. Plot a histogram — it should look uniform.

In [ ]:
# (a) Single die roll histogram
np.random.seed(42)  # For reproducibility

single = # YOUR CODE HERE — np.random.randint(1, 7, size=10000)

plt.hist(single, bins=np.arange(0.5, 7.5, 1), density=True, edgecolor='black', alpha=0.7)
plt.xlabel('Die Face')
plt.ylabel('Relative Frequency')
plt.title('Single Die Roll (should look uniform)')
plt.show()

**(b)** Simulate the sum of 2 dice, 10,000 times. Plot the histogram — it should look triangular.

> **Hint:** `np.random.randint(1, 7, size=(10000, 2))` creates a 10000×2 array. Then `.sum(axis=1)` sums each row (each "experiment").

In [ ]:
# (b) Sum of 2 dice
two_dice = # YOUR CODE HERE

plt.hist(two_dice, bins=np.arange(1.5, 13.5, 1), density=True, edgecolor='black', alpha=0.7)
plt.title('Sum of 2 Dice')
plt.show()

**(c)** Create a 2×2 subplot grid for sums of 1, 2, 5, and 30 dice. Observe how the distribution approaches a Gaussian.

In [ ]:
# (c) CLT visualisation: 2x2 subplot grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
n_dice_list = [1, 2, 5, 30]

for ax, n_dice in zip(axes.flatten(), n_dice_list):
    # YOUR CODE HERE — simulate sums, plot histogram on ax
    ax.set_title(f'Sum of {n_dice} Dice')
    ax.set_xlabel('Sum')
    ax.set_ylabel('Density')

plt.suptitle('Central Limit Theorem: Sums Approach a Gaussian', fontsize=14)
plt.tight_layout()
plt.show()

**(d)** For the 30-dice sum: overlay a Gaussian PDF with matching mean and standard deviation.

> **Hint:** Compute `sample_mean = np.mean(sums_30)` and `sample_std = np.std(sums_30)`, then plot `stats.norm.pdf(x_fit, loc=sample_mean, scale=sample_std)` on top of the histogram.

In [ ]:
# (d) 30-dice sum with Gaussian overlay
sums_30 = np.random.randint(1, 7, size=(10000, 30)).sum(axis=1)
sample_mean = np.mean(sums_30)
sample_std = np.std(sums_30)

plt.hist(sums_30, bins=30, density=True, edgecolor='black', alpha=0.5, label='Simulated')

# YOUR CODE HERE — overlay a Gaussian PDF curve

plt.xlabel('Sum of 30 Dice')
plt.title('CLT: 30 Dice Sum ≈ Gaussian')
plt.legend()
plt.show()

---

## Section 4: Bayes' Theorem `[CORE]` — 20 min

### Exercise 4.1: Medical Testing — Completing Exercise 1.4 from Lecture 1 (10 min)

**Context:** In Lecture 1 (Exercise 1.4), you computed the joint probabilities for a medical test:
- Disease prevalence: P(D) = 1%
- Test sensitivity: P(+|D) = 95%
- Test specificity: P(−|D^c) = 90%

You found P(+ ∩ D) = 0.0095 and P(+ ∩ D^c) = 0.099. Now, with Bayes' theorem: **if you test positive, what's the probability you actually have the disease?**

**(a)** Define the known quantities as Python variables.

In [ ]:
# (a) Define the given values
p_disease = 0.01                                  # P(D) — prevalence (prior)
p_pos_given_disease = 0.95                         # P(+|D) — sensitivity (likelihood)
p_neg_given_healthy = 0.90                         # P(-|D^c) — specificity
p_pos_given_healthy = 1 - p_neg_given_healthy      # P(+|D^c) — false positive rate

**(b)** Compute the evidence P(+) using the **Law of Total Probability**.

> **Recall:** P(+) = P(+|D)·P(D) + P(+|D^c)·P(D^c)

In [ ]:
# (b) Law of Total Probability
p_positive = # YOUR CODE HERE

print(f"P(positive test) = {p_positive:.4f}")

**(c)** Apply **Bayes' theorem** to compute P(D|+).

> **Recall:** P(D|+) = P(+|D) · P(D) / P(+)

In [ ]:
# (c) Bayes' theorem
p_disease_given_pos = # YOUR CODE HERE

print(f"P(Disease | Positive Test) = {p_disease_given_pos:.4f}")
print(f"That's only {p_disease_given_pos*100:.1f}% — even with a 'good' test!")

**(d)** Visualise: Left panel — bar chart comparing prior P(D) vs posterior P(D|+). Right panel — breakdown of positive tests into true positives vs false positives.

In [ ]:
# (d) Visualise prior vs posterior and true/false positives
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Prior vs Posterior
labels = ['Prior P(D)', 'Posterior P(D|+)']
values = [p_disease, p_disease_given_pos]

# YOUR CODE HERE — bar chart on ax1

# Right: True vs False positives
true_pos = p_pos_given_disease * p_disease
false_pos = p_pos_given_healthy * (1 - p_disease)

# YOUR CODE HERE — bar chart on ax2

plt.tight_layout()
plt.show()

**(e)** **Parametric exploration.** Write a function `compute_posterior(prevalence, sensitivity, specificity)` that returns P(D|+). Then sweep prevalence from 0.001 to 0.50 and plot posterior vs prevalence.

> **Question:** At approximately what prevalence does a positive test become "more likely true than false" (posterior > 0.5)?

In [ ]:
# (e) Parametric exploration
def compute_posterior(prevalence, sensitivity, specificity):
    """Compute P(Disease | Positive Test) using Bayes' theorem."""
    # YOUR CODE HERE
    pass

prevalences = np.array([0.001, 0.01, 0.05, 0.10, 0.20, 0.50])
posteriors = np.array([compute_posterior(p, 0.95, 0.90) for p in prevalences])

plt.plot(prevalences, posteriors, 'bo-', linewidth=2, markersize=8)
plt.axhline(y=0.5, color='red', linestyle='--', label='50% threshold')
plt.xlabel('Disease Prevalence (Prior)')
plt.ylabel('P(Disease | Positive Test)')
plt.title('How Prevalence Affects the Posterior')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 4.2: Multiple Hypotheses — Which Factory Made This Part? (10 min)

**Context:** Three factories produce identical-looking electronic components:
- **Factory A:** 50% of all units, 3% defect rate
- **Factory B:** 30% of all units, 5% defect rate
- **Factory C:** 20% of all units, 2% defect rate

You pick a random component and discover it is **defective**. Which factory most likely produced it?

**(a)** Define the priors and likelihoods as NumPy arrays.

In [ ]:
# (a) Define priors and likelihoods
factories = ['A', 'B', 'C']
prior = # YOUR CODE HERE — array of production shares
likelihood = # YOUR CODE HERE — array of defect rates P(defective | factory)

**(b)** Compute the posterior using **vectorised** Bayes' theorem — one line of NumPy!

> **The Pattern:** `posterior = (likelihood * prior) / np.sum(likelihood * prior)`
>
> This one-line vectorised Bayesian update is the computational heart of every Bayesian algorithm you'll write in this course.

In [ ]:
# (b) Vectorised Bayes' theorem
evidence = # YOUR CODE HERE — P(defective) via Law of Total Probability
posterior = # YOUR CODE HERE — Bayes' theorem

for f, pr, post in zip(factories, prior, posterior):
    print(f"Factory {f}: Prior = {pr:.2f}  →  Posterior = {post:.3f}")

**(c)** Create a 1×3 subplot: Prior → Likelihood → Posterior bar charts.

In [ ]:
# (c) Visualise: Prior → Likelihood → Posterior
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

# YOUR CODE HERE — bar charts on ax1 (prior), ax2 (likelihood), ax3 (posterior)

plt.suptitle("Bayes' Theorem: Prior × Likelihood → Posterior", fontsize=13)
plt.tight_layout()
plt.show()

**(d)** Which factory is most likely? Note that Factory A has the highest posterior despite a lower defect rate than B — the prior (production volume) dominates.

> **Looking Ahead:** The pattern `posterior = (likelihood * prior) / evidence` is the computational core of every Bayesian update in this course. You will use it for occupancy grids, sensor models, and belief propagation.

---

## Section 5: Recursive Bayesian Updates `[CORE]` — 25 min

This is the most important section of the lab. The posterior-becomes-prior loop is the engine behind occupancy grid mapping, Kalman filtering, and every SLAM algorithm.

### Exercise 5.1: Coin Bias Estimation — Sequential Updates (15 min)

**Context:** You find a coin and suspect it might be biased. You'll flip it repeatedly and update your belief about the bias after each flip.

**(a)** Discretise the hypothesis space into 101 values and define a uniform prior.

In [ ]:
# (a) Hypothesis space and uniform prior
theta = np.linspace(0, 1, 101)      # 101 possible bias values: 0.00, 0.01, ..., 1.00
prior = np.ones(101) / 101           # Uniform prior — no initial preference

plt.plot(theta, prior, 'b-', linewidth=2)
plt.xlabel('Bias θ')
plt.ylabel('P(θ)')
plt.title('Prior: Uniform (No Information)')
plt.ylim(0, 0.05)
plt.show()

**(b)** Define the likelihood function.

> For a coin with bias θ: P(Heads | θ) = θ,  P(Tails | θ) = 1 − θ

In [ ]:
# (b) Likelihood function
def likelihood(outcome, theta):
    """P(outcome | theta) for each value of theta."""
    # YOUR CODE HERE — return theta for 'H', return 1-theta for 'T'
    pass

**(c)** Perform one Bayesian update after observing Heads. Plot the posterior overlaid on the prior.

In [ ]:
# (c) One Bayesian update
lik = likelihood('H', theta)
unnormalized = # YOUR CODE HERE — multiply likelihood by prior
posterior = # YOUR CODE HERE — normalise

plt.plot(theta, prior, 'b--', label='Prior', alpha=0.5)
plt.plot(theta, posterior, 'r-', linewidth=2, label='Posterior (after 1 H)')
plt.xlabel('Bias θ')
plt.ylabel('P(θ)')
plt.title('One Bayesian Update')
plt.legend()
plt.show()

**(d)** Perform **20 sequential updates** from a biased coin (true bias = 0.7).

For each flip:
1. Compute the likelihood for the observed outcome
2. Multiply by the current belief (element-wise)
3. Normalise to get the new posterior

The posterior becomes the prior for the next flip!

Create a 2×3 subplot showing the posterior after flips 1, 3, 5, 10, 15, and 20.

In [ ]:
# (d) Sequential Bayesian updates
np.random.seed(42)
flips = np.random.choice(['H', 'T'], size=20, p=[0.7, 0.3])
print(f"Flip sequence: {''.join(flips)}")

belief = prior.copy()
means = []
stds = []

snapshot_flips = [1, 3, 5, 10, 15, 20]
snapshots = {}

for i, outcome in enumerate(flips):
    # Step 1: Compute likelihood for this outcome
    lik = # YOUR CODE HERE

    # Step 2: Multiply likelihood by current belief (unnormalised posterior)
    unnormalized = # YOUR CODE HERE

    # Step 3: Normalise to get proper posterior
    belief = # YOUR CODE HERE

    # Track statistics
    mean_i = np.sum(theta * belief)
    std_i = np.sqrt(np.sum(theta**2 * belief) - mean_i**2)
    means.append(mean_i)
    stds.append(std_i)

    if (i + 1) in snapshot_flips:
        snapshots[i + 1] = belief.copy()

# Plot the evolution
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, n_flip in zip(axes.flatten(), snapshot_flips):
    ax.plot(theta, snapshots[n_flip], 'r-', linewidth=2)
    ax.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='True bias = 0.7')
    ax.set_title(f'After {n_flip} flip(s)')
    ax.set_xlabel('Bias θ')
    ax.set_ylabel('P(θ)')
    ax.legend(fontsize=8)

plt.suptitle('Recursive Bayesian Updates: Posterior Sharpens With More Data', fontsize=14)
plt.tight_layout()
plt.show()

**(e)** Plot the convergence: posterior mean (should approach 0.7) and standard deviation (should decrease).

In [ ]:
# (e) Convergence plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, 21), means, 'bo-', markersize=5)
ax1.axhline(y=0.7, color='green', linestyle='--', label='True bias = 0.7')
ax1.set_xlabel('Flip Number')
ax1.set_ylabel('Posterior Mean')
ax1.set_title('Mean Converges to True Bias')
ax1.legend()

ax2.plot(range(1, 21), stds, 'ro-', markersize=5)
ax2.set_xlabel('Flip Number')
ax2.set_ylabel('Posterior Std Dev')
ax2.set_title('Uncertainty Decreases With More Data')

plt.tight_layout()
plt.show()

> **Looking Ahead:** The posterior-becomes-prior loop you just wrote is exactly what the SLAM system does every time the robot takes a new laser scan. Each scan updates the map, which becomes the prior for the next scan.

### Exercise 5.2: Weather Prediction — Multi-State Bayesian Updates (10 min)

**Context:** A weather system can be in one of three states: **Sunny**, **Cloudy**, or **Rainy**. Each day, you observe whether people are carrying umbrellas — that's your "sensor." You want to infer the true weather from umbrella observations.

**(a)** Define the problem: states, prior belief, and likelihoods.

In [ ]:
# (a) Define the problem
states = ['Sunny', 'Cloudy', 'Rainy']
prior = np.array([0.5, 0.3, 0.2])      # Initial belief

# Likelihood: P(umbrella | weather)
p_umbrella = np.array([0.1, 0.4, 0.9])
p_no_umbrella = 1 - p_umbrella

**(b)** Perform 7 sequential updates for the observation sequence: umbrella, umbrella, no umbrella, umbrella, no umbrella, umbrella, umbrella.

> **Hint:** The update is the same vectorised pattern: `belief = (lik * belief) / np.sum(lik * belief)`. Choose `lik = p_umbrella` or `lik = p_no_umbrella` based on the observation.

In [ ]:
# (b) Sequential updates
observations = ['U', 'U', 'N', 'U', 'N', 'U', 'U']  # U=umbrella, N=no umbrella
obs_labels = {'U': 'Umbrella', 'N': 'No umbrella'}

belief = prior.copy()
history = [belief.copy()]

for obs in observations:
    # YOUR CODE HERE — select likelihood, compute unnormalised posterior, normalise
    lik = # p_umbrella if obs == 'U' else p_no_umbrella
    unnormalized = # YOUR CODE HERE
    belief = # YOUR CODE HERE

    history.append(belief.copy())
    print(f"Observed {obs_labels[obs]:12s} → "
          f"Sunny={belief[0]:.3f}, Cloudy={belief[1]:.3f}, Rainy={belief[2]:.3f}")

**(c)** Plot the belief evolution as a multi-line chart.

In [ ]:
# (c) Belief evolution plot
history = np.array(history)  # shape: (8, 3)

plt.figure(figsize=(10, 5))
for i, state in enumerate(states):
    plt.plot(range(8), history[:, i], 'o-', linewidth=2, markersize=6, label=state)

plt.xlabel('Observation Number (0 = prior)')
plt.ylabel('Belief Probability')
plt.title('Belief Evolution Over 7 Observations')
plt.xticks(range(8), ['Prior'] + [f'Obs {j+1}' for j in range(7)], rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**(d)** What is the MAP estimate (most likely weather state) after all 7 observations?

> **Hint:** `np.argmax(array)` returns the index of the largest element.

In [ ]:
# (d) MAP estimate
map_index = # YOUR CODE HERE
print(f"MAP estimate: {states[map_index]} (P = {belief[map_index]:.3f})")

---

## Section 6: Log-Odds & Grid-Based Bayesian Updates `[CHALLENGE]` — 15 min

This section bridges directly to the SLAM occupancy grid you'll build in Week 5.

### Exercise 6.1: The Log-Odds Transform (7 min)

**Context:** In robotics, we work with **log-odds** instead of raw probabilities:

$$l(p) = \log\left(\frac{p}{1 - p}\right)$$

This transforms [0, 1] probabilities into (−∞, +∞) values, and turns multiplicative Bayesian updates into **simple addition**.

**(a)** Implement the log-odds transform and its inverse.

> **Formulas:**
> - Probability → Log-odds: `l = np.log(p / (1 - p))`
> - Log-odds → Probability: `p = 1 / (1 + np.exp(-l))`
> - Use `np.clip(p, 1e-10, 1 - 1e-10)` to avoid log(0) or division by zero

In [ ]:
# (a) Implement log-odds functions
def prob_to_log_odds(p):
    """Convert probability to log-odds: l = log(p / (1 - p))"""
    # YOUR CODE HERE
    pass

def log_odds_to_prob(l):
    """Convert log-odds to probability: p = 1 / (1 + exp(-l))"""
    # YOUR CODE HERE
    pass

# Verify round-trip
test_values = [0.1, 0.25, 0.5, 0.75, 0.9]
for p in test_values:
    recovered = log_odds_to_prob(prob_to_log_odds(p))
    print(f"p = {p:.2f} → log-odds = {prob_to_log_odds(p):+.4f} → recovered p = {recovered:.6f}")

**(b)** Plot the log-odds function. Add reference lines at key values:
- l = 0 corresponds to p = 0.5 ("unknown")
- l ≈ +2.2 corresponds to p ≈ 0.9 ("likely occupied")
- l ≈ −2.2 corresponds to p ≈ 0.1 ("likely free")

In [ ]:
# (b) Plot the log-odds transform
p_range = np.linspace(0.01, 0.99, 200)

# YOUR CODE HERE — plot log-odds curve with reference lines and annotations

plt.xlabel('Probability p')
plt.ylabel('Log-Odds l(p)')
plt.title('The Log-Odds Transform')
plt.grid(True, alpha=0.3)
plt.show()

**(c)** Demonstrate that log-odds updates are **additive**. Start with p = 0.5 (unknown). Apply two "occupied" observations with P(occupied | measurement) = 0.85.

Show that both methods give the same result:
- **Method 1 (probability):** multiply and normalise twice
- **Method 2 (log-odds):** just add `prob_to_log_odds(0.85)` twice, then convert back

In [ ]:
# (c) Additive property of log-odds

# Method 1: Probability-based updates (multiplicative)
p = 0.5
for _ in range(2):
    p_occ = 0.85 * p
    p_free = 0.15 * (1 - p)
    p = p_occ / (p_occ + p_free)
print(f"Probability method: p = {p:.6f}")

# Method 2: Log-odds (additive)
# YOUR CODE HERE — start at log_odds of 0.5, add log_odds of 0.85 twice, convert back

# print(f"Log-odds method:    p = {p_logodds:.6f}")
# print(f"\nBoth methods agree: {np.isclose(p, p_logodds)}")

### Exercise 6.2: 2D Grid Bayesian Updates with Log-Odds (8 min)

**Context:** Imagine a simplified 10×10 grid representing a room floor plan. Each cell can be **occupied** (furniture/wall) or **free** (empty space). We start with no information (all cells at 50% probability, log-odds = 0).

**(a)** Create the grid and define update values.

In [ ]:
# (a) Initialise grid and update values
grid = np.zeros((10, 10), dtype=np.float32)  # All cells start at log-odds = 0 (p = 0.5)

l_occ = prob_to_log_odds(0.85)    # "Sensor says occupied" → positive log-odds
l_free = prob_to_log_odds(0.40)   # "Sensor says free" → negative log-odds
l_max = 5.0    # Clamp to prevent over-confidence
l_min = -5.0

print(f"l_occ  = {l_occ:+.3f}  (one 'occupied' observation adds this)")
print(f"l_free = {l_free:+.3f}  (one 'free' observation adds this)")

**(b)** Simulate observations and create snapshots. A "wall" runs along **row 3, columns 2–7** and a "corridor" runs along **row 5, columns 2–7**. Create snapshots after 1, 5, and 20 observations.

> **Hint:** Use slice assignment: `grid[3, 2:8] += l_occ` updates a row of cells in one operation. Then clamp with `np.clip()`.

In [ ]:
# (b) Simulate observations and store snapshots
snapshots = {}

for n_obs in [1, 5, 20]:
    grid = np.zeros((10, 10), dtype=np.float32)  # Reset

    for _ in range(n_obs):
        # YOUR CODE HERE
        # Wall: row 3, columns 2-7 get "occupied" observations (add l_occ, then clip)
        # Corridor: row 5, columns 2-7 get "free" observations (add l_free, then clip)
        pass

    snapshots[n_obs] = grid.copy()

**(c)** Convert to probabilities and display as a 1×3 heatmap showing the evolution.

The wall should appear **red** (high P(occupied)), the corridor **green** (low P(occupied)), and unobserved cells **yellow** (P = 0.5).

> **Hint:** Use `plt.imshow(prob_grid, cmap='RdYlGn_r', vmin=0, vmax=1, origin='lower')`.
> The `origin='lower'` makes row 0 appear at the bottom (standard map convention).

In [ ]:
# (c) Heatmap visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_obs in zip(axes, [1, 5, 20]):
    prob_grid = log_odds_to_prob(snapshots[n_obs])

    # YOUR CODE HERE — imshow with cmap='RdYlGn_r', vmin=0, vmax=1, origin='lower'
    ax.set_title(f'After {n_obs} Observation(s)')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

# YOUR CODE HERE — add a shared colorbar

plt.suptitle('2D Grid Bayesian Updates with Log-Odds', fontsize=14)
plt.tight_layout()
plt.show()

> **Looking Ahead:** The grid you just built is a miniature version of the `OccupancyGrid` class in `occupancy_grid.py`. The actual SLAM code uses the same `np.zeros()` initialisation, the same `+=` log-odds updates, and the same `np.clip()` clamping you practised here — just on a 1000×700 grid with real laser scan data.

---

## Stretch Goal: Gaussian Bayesian Update `[STRETCH]`

**Only attempt this if you have finished all previous sections.**

When both the prior and the likelihood are Gaussian, the posterior is also Gaussian — computable in closed form. This is the core of the **Kalman filter**.

**Update formulas (from Lecture 2, Sec 2.5):**

$$\mu_n = \frac{\sigma_m^2 \cdot \mu_p + \sigma_p^2 \cdot z}{\sigma_p^2 + \sigma_m^2}$$

$$\sigma_n^2 = \frac{\sigma_p^2 \cdot \sigma_m^2}{\sigma_p^2 + \sigma_m^2}$$

**(a)** A temperature sensor measures a room. Prior: N(22.0, 3.0²)°C. Measurement: z = 24.5°C, σ_m = 1.5°C. Compute the posterior.

In [ ]:
# (a) Gaussian Bayesian update
mu_p, sigma_p = 22.0, 3.0    # Prior
z, sigma_m = 24.5, 1.5        # Measurement

sigma_p2 = sigma_p**2
sigma_m2 = sigma_m**2

# YOUR CODE HERE — compute mu_n and sigma_n using the formulas above
mu_n = # YOUR CODE HERE
sigma_n2 = # YOUR CODE HERE
sigma_n = np.sqrt(sigma_n2)

print(f"Prior:       μ = {mu_p:.2f}°C,  σ = {sigma_p:.2f}°C")
print(f"Measurement: z = {z:.2f}°C,  σ = {sigma_m:.2f}°C")
print(f"Posterior:   μ = {mu_n:.2f}°C,  σ = {sigma_n:.2f}°C")
print(f"\nPosterior is closer to measurement (more precise source gets more weight)")
print(f"Posterior uncertainty ({sigma_n:.2f}) < both prior ({sigma_p:.2f}) and measurement ({sigma_m:.2f})")

**(b)** Plot all three Gaussians (prior, measurement/likelihood, posterior) on the same axes.

In [ ]:
# (b) Plot prior, likelihood, and posterior
x = np.linspace(15, 30, 200)

plt.figure(figsize=(10, 5))

# YOUR CODE HERE — plot three Gaussians with labels and legend

plt.xlabel('Temperature (°C)')
plt.ylabel('Probability Density')
plt.title('Gaussian Bayesian Update')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**(c)** Perform 5 sequential measurements: z = [24.5, 23.0, 25.0, 24.2, 23.8], each with σ_m = 1.5°C. Plot how the mean converges and the standard deviation shrinks.

In [ ]:
# (c) Sequential Gaussian updates
measurements = [24.5, 23.0, 25.0, 24.2, 23.8]
sigma_m = 1.5

mu, sigma = 22.0, 3.0  # Start with original prior
mus = [mu]
sigmas = [sigma]

for z in measurements:
    # YOUR CODE HERE — apply the Gaussian update formulas
    pass
    mus.append(mu)
    sigmas.append(sigma)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(6), mus, 'bo-', markersize=8)
ax1.axhline(y=np.mean(measurements), color='green', linestyle='--', label='Mean of measurements')
ax1.set_xlabel('Update Number (0 = prior)')
ax1.set_ylabel('Posterior Mean (°C)')
ax1.set_title('Mean Converges to True Value')
ax1.legend()

ax2.plot(range(6), sigmas, 'ro-', markersize=8)
ax2.set_xlabel('Update Number (0 = prior)')
ax2.set_ylabel('Posterior Std Dev (°C)')
ax2.set_title('Uncertainty Decreases With Each Measurement')

plt.tight_layout()
plt.show()

---

## Summary: What You Learned Today

| Section | Probability Concept | Python Skill | SLAM Connection |
|---------|---------------------|-------------|------------------|
| 1 | Mean, std dev | `np.array`, `plt.bar`, `plt.fill_between` | Basic toolkit |
| 2 | PMF, joint/marginal/conditional | `np.sum(axis=)`, `plt.imshow` | Occupancy grid visualisation |
| 3 | Gaussian, CLT | `scipy.stats.norm`, `plt.hist` | Sensor noise modelling |
| 4 | Bayes' theorem | Vectorised `lik * prior / evidence` | Core Bayesian update |
| 5 | Recursive Bayes, MAP | Update loops, `np.argmax` | Sequential sensor fusion |
| 6 | Log-odds, 2D grids | `np.log`, `np.exp`, `np.clip`, `imshow` | Occupancy grid mapping |
| Stretch | Gaussian updates | Kalman filter formulas | SLAM state estimation |